# Stage 1 — Synthetic SFT Warmup

**Goal:** Teach Llama-3.2-1B to output valid negotiation tool calls and basic negotiation grammar.  
**Runtime:** ~30 min on Colab T4.  
**Output:** SFT checkpoint pushed to HF Hub.

Run cells top-to-bottom. Colab T4 free tier is sufficient.

In [ ]:
# Install deps
!pip install -q unsloth trl datasets openai

In [ ]:
import os, json, random
from pathlib import Path

# ── Config ──────────────────────────────────────────────────────────
BASE_MODEL     = "meta-llama/Llama-3.2-1B"
HF_REPO        = "YOUR_HF_USERNAME/agentgrid-sft"  # <- set your HF username
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")  # for trace generation
N_TRACES       = 2000
LORA_R         = 16
MAX_SEQ_LEN    = 2048
EPOCHS         = 2
OUTPUT_DIR     = "/content/sft_output"
DATA_PATH      = "/content/sft_data.jsonl"
# ────────────────────────────────────────────────────────────────────

## Step 1: Generate synthetic negotiation traces

Uses GPT-4o to generate 2000 realistic transcripts of 3-agent energy negotiation.

In [ ]:
SYSTEM_PROMPT = """
You are generating training data for a 3-agent energy negotiation environment.
Agents A, B, C each have: a battery (0-1), a pending task, and a reputation score.
Each turn, each agent calls exactly ONE of these tools:
  broadcast(agent_id, message)  — max 200 chars, free text
  make_offer(agent_id, to, give_type, give_amount, want_type, want_amount)
  accept_offer(agent_id, offer_id)
  execute_task(agent_id)
  renege(agent_id, offer_id)
  idle(agent_id)

Generate a realistic 8-12 step transcript showing agents negotiating energy trades.
Agents should reason about trust (reputation), urgency, and battery levels.
Format each turn as a JSON object with fields: step, agent, tool, kwargs, rationale.
Output a JSON array.
"""

def generate_trace(client, scenario_seed: int) -> list[dict]:
    random.seed(scenario_seed)
    batteries = {a: round(random.uniform(0.2, 1.0), 2) for a in "ABC"}
    urgencies = {a: round(random.uniform(0.2, 1.0), 2) for a in "ABC"}
    user_msg = (
        f"Scenario: batteries={batteries}, urgencies={urgencies}. "
        "Generate a negotiation transcript (8-12 steps)."
    )
    try:
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_msg},
            ],
            response_format={"type": "json_object"},
            max_tokens=1200,
        )
        return json.loads(resp.choices[0].message.content).get("turns", [])
    except Exception as e:
        print(f"  trace {scenario_seed} failed: {e}")
        return []

if OPENAI_API_KEY:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    traces = []
    for i in range(N_TRACES):
        if i % 100 == 0:
            print(f"Generating trace {i}/{N_TRACES}...")
        trace = generate_trace(client, i)
        if trace:
            traces.append(trace)
    print(f"Generated {len(traces)} traces.")
else:
    print("OPENAI_API_KEY not set — using stub traces for testing.")
    traces = [[{"step": 1, "agent": "A", "tool": "broadcast",
                "kwargs": {"message": "Battery 0.74. Open to trades."},
                "rationale": "Announce state."}]] * 50

In [ ]:
# Convert traces to SFT format: (observation_prompt, tool_call_completion)
def trace_to_sft_examples(trace: list[dict]) -> list[dict]:
    examples = []
    for turn in trace:
        prompt = (
            f"You are Agent {turn.get('agent', 'A')}.\n"
            f"Step {turn.get('step', 1)}. Choose ONE action tool.\n"
            f"Rationale context: {turn.get('rationale', '')}"
        )
        kwargs = turn.get('kwargs', {})
        completion = json.dumps({"tool": turn.get('tool', 'idle'), "kwargs": kwargs})
        examples.append({"prompt": prompt, "completion": completion})
    return examples

sft_data = [ex for trace in traces for ex in trace_to_sft_examples(trace)]
random.shuffle(sft_data)

with open(DATA_PATH, "w") as f:
    for ex in sft_data:
        f.write(json.dumps(ex) + "\n")

print(f"Saved {len(sft_data)} SFT examples to {DATA_PATH}")

## Step 2: SFT with Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=torch.float16,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_R,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing=True,
)
print("Model loaded.")

In [ ]:
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

dataset = load_dataset("json", data_files=DATA_PATH, split="train")

def format_example(ex):
    return {"text": f"<s>[INST] {ex['prompt']} [/INST] {ex['completion']} </s>"}

dataset = dataset.map(format_example)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        save_strategy="epoch",
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
    ),
)

trainer.train()

## Step 3: Evaluate JSON validity rate

In [ ]:
FastLanguageModel.for_inference(model)

test_prompt = (
    "<s>[INST] You are Agent A. Step 5. Battery: 0.62. Urgency: 0.81. "
    "Agent B has reputation 0.91. Choose ONE action tool. [/INST]"
)

valid_count = 0
N_EVAL = 50
for _ in range(N_EVAL):
    inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=80, do_sample=True, temperature=0.7)
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    completion = text[len(test_prompt):].strip()
    try:
        json.loads(completion)
        valid_count += 1
    except json.JSONDecodeError:
        pass

validity_rate = valid_count / N_EVAL
print(f"JSON validity rate: {validity_rate:.1%}")
assert validity_rate >= 0.7, f"Validity rate {validity_rate:.1%} too low — check traces."

## Step 4: Push checkpoint to HF Hub

In [ ]:
from huggingface_hub import login
login()  # paste HF token when prompted

model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)
print(f"Checkpoint saved to hf.co/{HF_REPO}")